# CHUẨN BỊ DỮ LIỆU
Vì tập dữ liệu quá lớn, khó thao tác trên trên laptop cá nhân nên phần chuẩn bị tập dữ liệu được thực hiện trên Kaggle.

Đường link các tập dữ liệu:
1. celeb_dataset: [CelebFaces Attributes (CelebA) Dataset](https://www.kaggle.com/datasets/jessicali9530/celeba-dataset)
2. places_dataset: [Places365](https://www.kaggle.com/datasets/benjaminkz/places365)
3. train_mask: [NVIDIA Irregular Mask Dataset: Testing Set](https://www.dropbox.com/scl/fi/l0b4dvae2pra70qpg7r38/test_mask.zip?rlkey=4kfby3x61yssw2nyi7qsah2wh&e=1&dl=0)
4. test_mask: [NVIDIA Irregular Mask Dataset: Training Set](https://www.dropbox.com/scl/fi/ffyha2ymdx7ngpa2vs5j2/irregular_mask.zip?rlkey=535a7nsitdn48lxhb6hbvexbn&e=1&dl=0)

In [ ]:
from collections import defaultdict
import os
import random
import shutil
import glob

In [ ]:
celeb_dataset = '/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba'
places_train_dataset = '/kaggle/input/datasets/nickj26/places2-mit-dataset/train_256_places365standard/data_256'
places_test_dataset = '/kaggle/input/datasets/nickj26/places2-mit-dataset/test_256/test_256'


trainset = '/kaggle/working/trainset'
testset = '/kaggle/working/testset'

os.makedirs(trainset, exist_ok=True)
os.makedirs(testset, exist_ok=True)

In [ ]:
def get_balanced_image_paths(directory, total_samples):
    class_images = defaultdict(list)
    n=1
    
    for root, _, files in os.walk(directory):
        print(n)
        n+= 1
        valid_imgs = [os.path.join(root, f) for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if valid_imgs:
            class_images[root].extend(valid_imgs)
            
    num_classes = len(class_images)
    if num_classes == 0:
        return []
    
    samples_per_class = total_samples // num_classes
    remainder = total_samples % num_classes
    
    sampled_paths = []
    
    for i, (cls_dir, imgs) in enumerate(class_images.items()):
        k = samples_per_class + (1 if i < remainder else 0)

        k = min(k, len(imgs))
        
        sampled_paths.extend(random.sample(imgs, k))
        
    return sampled_paths

train_places = get_balanced_image_paths(places_train_dataset, total_samples=180000)
random.shuffle(train_places)

In [ ]:
test_places = []
for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
    test_places.extend(glob.glob(os.path.join(places_test_dataset, ext)))
random.seed(42)
random.shuffle(test_places)

test_places = test_places[:20000]

In [ ]:
celeb_image = []
for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
    celeb_image.extend(glob.glob(os.path.join(celeb_dataset, ext)))
random.seed(42)
random.shuffle(celeb_image)

split_idx = int(0.9* len(celeb_image))
train_celeb = celeb_image[:split_idx]
test_celeb = celeb_image[split_idx:]

In [ ]:
def rename_img(file_list, destination_dir, prefix=""):
    for i, img_path in enumerate(file_list):
        base_name = os.path.basename(img_path)
        
        if prefix == "":
            new_name = f"{i}_{base_name}"
        else:
            new_name = f"{prefix}_{i}_{base_name}"
            
        dst_path = os.path.join(destination_dir, new_name)
        shutil.copy(img_path, dst_path)

rename_img(train_places, trainset, prefix="places")
rename_img(train_celeb, trainset, prefix="celeb")

rename_img(test_places, testset, prefix="")
rename_img(test_celeb, testset, prefix="celeb")

In [ ]:
shutil.make_archive('/kaggle/working/train_dataset', 'zip', trainset)
shutil.make_archive('/kaggle/working/test_dataset', 'zip', testset)

shutil.rmtree(trainset)
shutil.rmtree(testset)